# memopro CUDA check (pre-release gate V2, docs/research/0044)

Checks on a real NVIDIA GPU what the Mac could not: `doctor` with CUDA, census coverage on CUDA,
and hibernation (`host`, `source`, backward in flight) with bit-exact restores.

1. **Runtime -> Change runtime type -> T4 GPU**.
2. Upload the Linux wheel (`memopro-0.1.0a1-cp311-abi3-manylinux_*.whl`, from the *Release*
   workflow's manual run) with the file panel on the left, into `/content`.
3. **Runtime -> Run all**. Copy the JSON printed by the last cell back to the conversation.

Every section records its own result; a failing section does not stop the others.
Run 2 (0045) also checks: `host` reports added host RAM as a negative `rss`, and census
counts the cuBLAS workspace (compare `framework_workspace` with `independent_workspace`).

In [ ]:
import glob, json, subprocess, sys, traceback

wheels = sorted(glob.glob("/content/memopro-*.whl"))
if wheels:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", wheels[-1]], check=True)
import memopro, torch  # noqa: E402

results = {"memopro": memopro.__version__, "torch": torch.__version__,
           "cuda": torch.cuda.is_available(),
           "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}


def section(name):
    def wrap(fn):
        try:
            results[name] = fn()
        except Exception as e:  # keep going: report everything
            results[name] = {"error": f"{type(e).__name__}: {e}", "trace": traceback.format_exc()[-800:]}
        print(name, "->", json.dumps(results[name], default=str)[:300])
    return wrap
print(results)

In [ ]:
@section("doctor")
def _():
    r = memopro.doctor()
    print(r.summary())
    cuda = [d for d in r.env.devices if d.kind == "cuda"]
    return {"cuda_devices": len(cuda), "device_budget": r.budget.device,
            "free_reported": cuda[0].available_bytes if cuda else None,
            "free_torch": torch.cuda.mem_get_info(0)[0] if torch.cuda.is_available() else None,
            "notes": r.warnings()}

In [ ]:
from transformers import GPT2Config, GPT2LMHeadModel

@section("census_cuda")
def _():
    if not torch.cuda.is_available():
        return "skipped: no CUDA"
    torch.manual_seed(0)
    m = GPT2LMHeadModel(GPT2Config(n_layer=4, n_embd=256, n_head=4, vocab_size=8000)).cuda()
    opt = torch.optim.AdamW(m.parameters())
    x = torch.randint(0, 8000, (4, 64), device="cuda")
    for _ in range(2):
        opt.zero_grad(); m(x, labels=x).loss.backward(); opt.step()
    # independent check of the 0045 F2 hypothesis: cuBLAS workspaces live in the allocator
    torch.cuda.synchronize()
    a0 = torch.cuda.memory_allocated()
    torch._C._cuda_clearCublasWorkspaces()
    independent_workspace = a0 - torch.cuda.memory_allocated()
    opt.zero_grad(); m(x, labels=x).loss.backward(); opt.step()  # workspace comes back
    with memopro.census.record(m, opt, mode="light") as c:
        opt.zero_grad(); m(x, labels=x).loss.backward(); opt.step()
    print(c.summary())
    j = c.to_json()
    return {"coverage_cuda": j["coverage_at_end"].get("cuda"),
            "unattributed_cuda": j["unclassified_at_end"].get("cuda"),
            "framework_workspace": j.get("framework_workspace"),
            "independent_workspace": independent_workspace,
            "bytes": {k: v["bytes"] for k, v in j["categories"].items()}}


In [ ]:
def cuda_mem():
    torch.cuda.synchronize(); torch.cuda.empty_cache()
    return torch.cuda.memory_reserved()

@section("hibernate_gpt2")
def _():
    if not torch.cuda.is_available():
        return "skipped: no CUDA"
    out = {}
    model = GPT2LMHeadModel.from_pretrained("openai-community/gpt2").cuda().eval()
    x = torch.randint(0, 50257, (1, 64), device="cuda")
    with torch.no_grad():
        ref = model(x).logits.float().cpu()
    for mode in ("host", "auto"):
        before = cuda_mem()
        h = memopro.hibernate.now(model, mode=mode)
        freed = before - cuda_mem()
        with torch.no_grad():
            got = model(x).logits.float().cpu()  # forward wakes it
        out[mode] = {"modes": h.bytes_by_mode(), "kept": {k: len(v) for k, v in h.kept.items()},
                     "cuda_reserved_freed": freed, "reclaimed_report": h.reclaimed,
                     "bit_exact": bool(torch.equal(got, ref)), "asleep_after_forward": h.asleep}
    return out

In [ ]:
@section("backward_in_flight")
def _():
    if not torch.cuda.is_available():
        return "skipped: no CUDA"
    torch.manual_seed(0)
    make = lambda: torch.nn.Sequential(torch.nn.Linear(512, 512), torch.nn.Tanh(), torch.nn.Linear(512, 1)).cuda()
    a, b = make(), make(); b.load_state_dict(a.state_dict())
    x = torch.randn(8, 512, device="cuda", requires_grad=True)
    b(x).sum().backward()
    loss = a(x).sum()
    h = memopro.hibernate.now(a, mode="host")
    loss.backward()
    return {"grads_equal": all(torch.equal(p.grad, q.grad) for p, q in zip(a.parameters(), b.parameters())),
            "kept": {k: len(v) for k, v in h.kept.items()}}

In [ ]:
print(json.dumps(results, indent=2, default=str))